In [ ]:
# 下载 Titanic 数据（Kaggle 环境额外装 fastai）
from pathlib import Path
import os

iskaggle = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '')
if iskaggle:
    path = Path('../input/titanic')
    !pip install -Uqq fastai
else:
    import zipfile,kaggle
    path = Path('titanic')
    kaggle.api.competition_download_cli(str(path))
    zipfile.ZipFile(f'{path}.zip').extractall(path)

In [ ]:
# 导入 fastai 表格模块，设显示格式，固定随机种子
from fastai.tabular.all import *

pd.options.display.float_format = '{:.2f}'.format
set_seed(42)

In [ ]:
# 读训练数据
df = pd.read_csv(path/'train.csv')

In [ ]:
# 特征工程：对数票价 / 甲板 / 家庭规模 / 是否独行 / 票号频次 / 称谓  # 修正：Title 行末尾误加了 .value_counts(dropna=False)，会把整列替换成计数结果、和 df 行对不齐
def add_features(df):
    df['LogFare'] = np.log1p(df['Fare'])
    df['Deck'] = df.Cabin.str[0].map(dict(A="ABC", B="ABC", C="ABC", D="DE", E="DE", F="FG", G="FG"))
    df['Family'] = df.SibSp+df.Parch
    df['Alone'] = df.Family==1
    df['TicketFreq'] = df.groupby('Ticket')['Ticket'].transform('count')
    df['Title'] = df.Name.str.split(', ', expand=True)[1].str.split('.', expand=True)[0]
    df['Title'] = df.Title.map(dict(Mr="Mr",Miss="Miss",Mrs="Mrs",Master="Master"))

add_features(df)

In [ ]:
# 划分训练 / 验证集
splits = RandomSplitter(seed=42)(df)

In [ ]:
# 用 TabularPandas 组织数据（类别列 / 连续列 / 预处理 / 标签）
dls = TabularPandas(
    df, splits=splits,
    procs = [Categorify, FillMissing, Normalize],
    cat_names=["Sex","Pclass","Embarked","Deck", "Title"],
    cont_names=['Age', 'SibSp', 'Parch', 'LogFare', 'Alone', 'TicketFreq', 'Family'],
    y_names="Survived", y_block = CategoryBlock(),
).dataloaders(path=".")

In [ ]:
# 建表格神经网络（两个隐藏层，各 10）
learn = tabular_learner(dls, metrics=accuracy, layers=[10,10])

In [ ]:
# 找合适的学习率
learn.lr_find(suggest_funcs=(slide, valley))

In [ ]:
# 训练 16 轮，lr=0.03
learn.fit(16, lr=0.03)

In [ ]:
# 读测试集并做同样的特征工程
tst_df = pd.read_csv(path/'test.csv')
tst_df['Fare'] = tst_df.Fare.fillna(0)
add_features(tst_df)

In [ ]:
# 把测试集包成 fastai 的 test dataloader
tst_dl = learn.dls.test_dl(tst_df)

In [ ]:
# 预测
preds,_ = learn.get_preds(dl=tst_dl)

In [ ]:
# 取“生还”概率>0.5 生成提交文件
tst_df['Survived'] = (preds[:,1]>0.5).int()
sub_df = tst_df[['PassengerId','Survived']]
sub_df.to_csv('sub.csv', index=False)

In [ ]:
# 看提交文件前几行
!head sub.csv

In [ ]:
# 定义 ensemble：重新训练一个模型并返回其预测
def ensemble():
    learn = tabular_learner(dls, metrics=accuracy, layers=[10,10])
    with learn.no_bar(),learn.no_logging(): learn.fit(16, lr=0.03)
    return learn.get_preds(dl=tst_dl)[0]

In [ ]:
# 训练 5 个模型
learns = [ensemble() for _ in range(5)]

In [ ]:
# 5 个模型的预测取平均（集成）
ens_preds = torch.stack(learns).mean(0)

In [ ]:
# 用集成结果生成提交文件
tst_df['Survived'] = (ens_preds[:,1]>0.5).int()
sub_df = tst_df[['PassengerId','Survived']]
sub_df.to_csv('ens_sub.csv', index=False)